In [ ]:
import zipfile
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/NSW")

### Data cleaning notebook

A notebook that runs the workflow to clean the three raw NSW datasets (temperature, total demand, forecast demand) and merges them into a single half hourly table that can be used for additional joins and modelling.

General cleaning, such as nulls and duplicates are addressed, additionally there is also a closest to merge approach with timestamps for temperature, given it is not in strict 30minute format.

#### 0. Data cleaning overview

![Data cleaning pipeline](../report/images/data_cleaning_diagram.png)

#### 1. Unzip the raw data

The data provided to us is all in zipped csvs, this step of the data cleaning is just unzipping the files. Temperature, total demand, and forecast demand. The forecast demand file is split into two parts (partaa and .partab), so I have decided to join those back into one zip first, then unzip all three.

In [49]:
# join the split forecastdemand zip into one file
forecast_zip = DATA_DIR / "forecastdemand_nsw.csv.zip"
if not forecast_zip.exists():
    parts = sorted(DATA_DIR.glob("forecastdemand_nsw.csv.zip.part*"))
    with open(forecast_zip, "wb") as out_file:
        for part in parts:
            out_file.write(part.read_bytes())

# unzip all three datasets
for zip_name in ["temperature_nsw.csv.zip", "totaldemand_nsw.csv.zip", "forecastdemand_nsw.csv.zip"]:
    csv_name = zip_name.replace(".zip", "")
    if not (DATA_DIR / csv_name).exists():
        with zipfile.ZipFile(DATA_DIR / zip_name) as zf:
            zf.extractall(DATA_DIR)

#### 2. Temperature

Half hourly readings from the weather station. Here I am just dropping duplicate rows and missing readings.

I have chosen to drop missing readings as given the vast amount of data, I think we have the leeway to drop them instead of trying to impute. 

In [50]:
temp_df = pd.read_csv(DATA_DIR / "temperature_nsw.csv", parse_dates=["DATETIME"], dayfirst=True)
temp_df.info()
temp_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 220326 entries, 0 to 220325
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   LOCATION     220326 non-null  str           
 1   DATETIME     220326 non-null  datetime64[us]
 2   TEMPERATURE  220326 non-null  float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 5.0 MB


,LOCATION,DATETIME,TEMPERATURE
0,Bankstown,2010-01-01 00:00:00,23.1
1,Bankstown,2010-01-01 00:01:00,23.1
2,Bankstown,2010-01-01 00:30:00,22.9
3,Bankstown,2010-01-01 00:50:00,22.7
4,Bankstown,2010-01-01 01:00:00,22.6


In [51]:
dropped_duplicates = temp_df[temp_df.duplicated(keep="first")]
dropped_missing = temp_df[temp_df["TEMPERATURE"].isna()]

print(f"duplicate rows dropped: {len(dropped_duplicates)}")
print(dropped_duplicates)
print()
print(f"missing TEMPERATURE rows dropped: {len(dropped_missing)}")
print(dropped_missing)

duplicate rows dropped: 13
         LOCATION            DATETIME  TEMPERATURE
19006   Bankstown 2011-01-01 00:00:00         21.0
34282   Bankstown 2011-10-10 10:30:00         18.9
34299   Bankstown 2011-10-10 18:30:00         16.1
34302   Bankstown 2011-10-10 19:30:00         15.5
38655   Bankstown 2012-01-01 00:00:00         15.4
58293   Bankstown 2013-01-01 00:00:00         21.0
78276   Bankstown 2014-01-01 00:00:00         20.4
97917   Bankstown 2015-01-01 00:00:00         20.9
117699  Bankstown 2016-01-01 00:00:00         16.9
137200  Bankstown 2017-01-01 00:00:00         22.6
157015  Bankstown 2018-01-01 00:00:00         22.4
176797  Bankstown 2019-01-01 00:00:00         22.3
196230  Bankstown 2020-01-01 00:00:00         19.4

missing TEMPERATURE rows dropped: 0
Empty DataFrame
Columns: [LOCATION, DATETIME, TEMPERATURE]
Index: []


In [52]:
# drop any duplicate rows and missing readings
temp_df = temp_df.drop_duplicates()
temp_df = temp_df.dropna(subset=["TEMPERATURE"])

#### 3. Total demand

Half hourly actual electricity demand for NSW. Here I am dropping duplicate timestamps, missing readings, and any non positive demand values.

There are actually no duplicates, missing values, or negative/zero readings in this data.


In [53]:
demand_df = pd.read_csv(DATA_DIR / "totaldemand_nsw.csv", parse_dates=["DATETIME"], dayfirst=True)
demand_df.info()
demand_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 196513 entries, 0 to 196512
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   DATETIME     196513 non-null  datetime64[us]
 1   TOTALDEMAND  196513 non-null  float64       
 2   REGIONID     196513 non-null  str           
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 4.5 MB


,DATETIME,TOTALDEMAND,REGIONID
0,2010-01-01 00:00:00,8038.00,NSW1
1,2010-01-01 00:30:00,7809.31,NSW1
2,2010-01-01 01:00:00,7483.69,NSW1
3,2010-01-01 01:30:00,7117.23,NSW1
4,2010-01-01 02:00:00,6812.03,NSW1


In [54]:
# one row per timestamp, drop missing and nonpositive demand
demand_df = demand_df.drop_duplicates(subset="DATETIME")
demand_df = demand_df.dropna(subset=["TOTALDEMAND"])
demand_df = demand_df[demand_df["TOTALDEMAND"] > 0]
demand_df = demand_df.sort_values("DATETIME").reset_index(drop=True)
demand_df.head()

,DATETIME,TOTALDEMAND,REGIONID
0,2010-01-01 00:00:00,8038.00,NSW1
1,2010-01-01 00:30:00,7809.31,NSW1
2,2010-01-01 01:00:00,7483.69,NSW1
3,2010-01-01 01:30:00,7117.23,NSW1
4,2010-01-01 02:00:00,6812.03,NSW1


#### 4. Forecast demand

This file is large (~740MB uncompressed), around 55 per timestamp here. Lead times range from ~14 minutes to ~1.5 days ahead of the target time, so this is a short range forecast, no such thing as a week or month ahead forecast in this file.

I only load the columns needed with compact dtypes to keep this manageable in memory, drop missing forecast values, then pull out three columns per target `DATETIME`: `forecast_closest` (shortest lead time — the freshest forecast available), `forecast_12hr_prior` (closest to 12 hours ahead), and `forecast_dayprior` (closest to 24 hours ahead).

There are actually no missing values in this file either.

In [55]:
forecast_df = pd.read_csv(
    DATA_DIR / "forecastdemand_nsw.csv",
    usecols=["FORECASTDEMAND", "LASTCHANGED", "DATETIME"],
    dtype={"FORECASTDEMAND": "float32"},
    parse_dates=["DATETIME", "LASTCHANGED"],  # already ISO (yyyy-mm-dd), no dayfirst needed
)
forecast_df = forecast_df.dropna(subset=["FORECASTDEMAND"])

forecast_df["LEADHRS"] = (forecast_df["DATETIME"] - forecast_df["LASTCHANGED"]).dt.total_seconds() / 3600
forecast_df["LEADHRS"].describe()

count    1.090602e+07
mean     1.496778e+01
std      9.347107e+00
min      2.338889e-01
25%      6.985000e+00
50%      1.398306e+01
75%      2.149875e+01
max      3.949056e+01
Name: LEADHRS, dtype: float64

In [56]:
def nearest_lead_forecast(df, target_hours, column_name):
    diff = (df["LEADHRS"] - target_hours).abs()
    nearest = df.assign(_diff=diff).sort_values("_diff").drop_duplicates(subset="DATETIME", keep="first")
    nearest = nearest.sort_values("DATETIME").reset_index(drop=True)
    return nearest[["DATETIME", "FORECASTDEMAND"]].rename(columns={"FORECASTDEMAND": column_name})


forecast_closest = nearest_lead_forecast(forecast_df, target_hours=0, column_name="forecast_closest")
forecast_12hr_prior = nearest_lead_forecast(forecast_df, target_hours=12, column_name="forecast_12hr_prior")
forecast_dayprior = nearest_lead_forecast(forecast_df, target_hours=24, column_name="forecast_dayprior")

forecast_df = forecast_closest.merge(forecast_12hr_prior, on="DATETIME").merge(forecast_dayprior, on="DATETIME")
forecast_df.head()

,DATETIME,forecast_closest,forecast_12hr_prior,forecast_dayprior
0,2010-01-01 00:00:00,7999.109863,7811.859863,7822.379883
1,2010-01-01 00:30:00,7596.209961,7612.629883,7715.680176
2,2010-01-01 01:00:00,7380.700195,7339.640137,7482.560059
3,2010-01-01 01:30:00,7022.049805,7009.549805,7129.319824
4,2010-01-01 02:00:00,6682.919922,6683.149902,6800.729980


#### 5. Merge and save

Temperature timestamps don't line up exactly with the half hourly demand timestamps, so I have used `merge_asof` to attach the nearest reading to each demand row. This does not create duplicates as merge_asof attaches exactly one nearest match per demand row (like a left join). This is much quicker than resampling and it means that we get coverage of data. 

Additionally you will see very temperature is assigned to a timestamp so the approach has worked well.

In [57]:
# clean up 
for name in ["temperature_nsw.csv", "totaldemand_nsw.csv", "forecastdemand_nsw.csv", "forecastdemand_nsw.csv.zip"]:
    path = DATA_DIR / name
    if path.exists():
        path.unlink()

In [58]:
cleaned_df = pd.merge_asof(demand_df, temp_df, on="DATETIME", direction="nearest")
cleaned_df = pd.merge_asof(cleaned_df, forecast_df, on="DATETIME", direction="nearest")
cleaned_df.info()
cleaned_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 196513 entries, 0 to 196512
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   DATETIME             196513 non-null  datetime64[us]
 1   TOTALDEMAND          196513 non-null  float64       
 2   REGIONID             196513 non-null  str           
 3   LOCATION             196513 non-null  str           
 4   TEMPERATURE          196513 non-null  float64       
 5   forecast_closest     196513 non-null  float32       
 6   forecast_12hr_prior  196513 non-null  float32       
 7   forecast_dayprior    196513 non-null  float32       
dtypes: datetime64[us](1), float32(3), float64(2), str(2)
memory usage: 9.7 MB


,DATETIME,TOTALDEMAND,REGIONID,LOCATION,TEMPERATURE,forecast_closest,forecast_12hr_prior,forecast_dayprior
0,2010-01-01 00:00:00,8038.00,NSW1,Bankstown,23.1,7999.109863,7811.859863,7822.379883
1,2010-01-01 00:30:00,7809.31,NSW1,Bankstown,22.9,7596.209961,7612.629883,7715.680176
2,2010-01-01 01:00:00,7483.69,NSW1,Bankstown,22.6,7380.700195,7339.640137,7482.560059
3,2010-01-01 01:30:00,7117.23,NSW1,Bankstown,22.5,7022.049805,7009.549805,7129.319824
4,2010-01-01 02:00:00,6812.03,NSW1,Bankstown,22.5,6682.919922,6683.149902,6800.729980


In [59]:
# coverage check: are there any gaps in the half hourly timeline?
full_range = pd.date_range(cleaned_df["DATETIME"].min(), cleaned_df["DATETIME"].max(), freq="30min")
missing_timestamps = full_range.difference(cleaned_df["DATETIME"])

print(f"expected half-hourly intervals: {len(full_range)}")
print(f"actual rows: {len(cleaned_df)}")
print(f"missing intervals: {len(missing_timestamps)}")
print(missing_timestamps[:20])

# missing values per column
print()
print("missing values per column:")
print(cleaned_df.isna().sum())

# duplicate rows
print()
print("fully duplicate rows:", cleaned_df.duplicated().sum())
print("duplicate DATETIME rows:", cleaned_df.duplicated(subset="DATETIME").sum())

# dtypes: one declared dtype per column, and one consistent python type within each column
print()
print(cleaned_df.dtypes)
print()
print("distinct python types found per column:")
print(cleaned_df.map(type).nunique())

expected half-hourly intervals: 196513
actual rows: 196513
missing intervals: 0
DatetimeIndex([], dtype='datetime64[us]', freq='30min')

missing values per column:
DATETIME               0
TOTALDEMAND            0
REGIONID               0
LOCATION               0
TEMPERATURE            0
forecast_closest       0
forecast_12hr_prior    0
forecast_dayprior      0
dtype: int64

fully duplicate rows: 0
duplicate DATETIME rows: 0

DATETIME               datetime64[us]
TOTALDEMAND                   float64
REGIONID                          str
LOCATION                          str
TEMPERATURE                   float64
forecast_closest              float32
forecast_12hr_prior           float32
forecast_dayprior             float32
dtype: object

distinct python types found per column:
DATETIME               1
TOTALDEMAND            1
REGIONID               1
LOCATION               1
TEMPERATURE            1
forecast_closest       1
forecast_12hr_prior    1
forecast_dayprior      1
dtype: int6

In [60]:
# given the file is reasonably sized ~10mb, I have chosen csv as I think everyone in the team is the most comfortable with this format
cleaned_df.to_csv(DATA_DIR / "nsw_cleaned.csv", index=False)